In [1]:
import numpy as np
from random import sample
from scipy.sparse import csgraph, csr_matrix, triu
from ase.neighborlist import NeighborList, natural_cutoffs
from ase.build.rotate import rotation_matrix_from_points
from ase.io import read, write

def shuffled(indices):
    return sample(list(indices), len(indices))

def nodematch(node, mapping):
    nl1.update(mol1[mapping])
    matches = set(nl0.get_neighbors(node)[0]) & set(nl1.get_neighbors(node)[0])
    mismatches0 = set(nl0.get_neighbors(node)[0]) - matches
    mismatches1 = set(nl1.get_neighbors(node)[0]) - matches
    return matches, mismatches0, mismatches1

def moldiff(mapping):
    nl1.update(mol1[mapping])
    matrix1 = nl1.get_connectivity_matrix()
    return csr_matrix.count_nonzero(triu(matrix0) != triu(matrix1))

def rmsd(mapping):
    p0 = mol0.get_positions()
    p1 = mol1[mapping].get_positions()
    R = rotation_matrix_from_points(p1.T, p0.T)
    p1 = np.dot(p1, R.T)
    return np.sqrt(3*((p1 - p0)**2).mean())

# Función recursiva
def minadjdiff(node, mapping, tracked, depth=0):
    # Encontrar los nodos vecinos apareados y mal apareados y calcular los descriptores
    matches, mismatches0, mismatches1 = nodematch(node, mapping)
    #debug: Imprimir información del paso
    #print(' '*4*depth, f'{n+1}/{mapping[n]+1}:', matches, mismatches0, mismatches1, moldiff(mapping), f'{rmsd(mapping):.4f}')
    # Agregar el nodo n al conjunto de nodos recorridos
    tracked.add(node)
    # Aplicar la función recursivamente para cada nodo vecino apareado
    for i in shuffled(matches):
        if i not in tracked:
            minadjdiff(i, mapping, tracked, depth + 1)
    # Run over every mismatches neighbor in reference molecule
    for i in shuffled(mismatches0):
        # which is not yet tracked
        if i not in tracked:
            # Run over every mismatches neighbor in working molecule
            for j in shuffled(mismatches1):
                # Si los nodos corresponden al mismo elemento
                if mol0.symbols[i] == mol1[mapping].symbols[j]:
                    # Hacer una copia local del mapeo y del recorrido
                    tracked_branch = tracked.copy()
                    mapping_branch = mapping.copy()
                    # Hacer el intercambio
                    mapping_branch[i], mapping_branch[j] = mapping_branch[j], mapping_branch[i]
                    # Aplicar la función recursivamente a la rama que nace en el nodo intercambiado
                    minadjdiff(i, mapping_branch, tracked_branch, depth + 1)
                    # Si el intercambio no fue rechazado
                    if moldiff(mapping_branch) < moldiff(mapping):
                        # Agregar todos los nodos de la rama a la lista de recorridos
                        tracked.update(tracked_branch)
                        # Actualizar el mapeo con los intercambios hechos en la rama
                        mapping[:] = mapping_branch[:]
                        # Remove matched neighbors from mismatches
                        mismatches0.remove(i)
                        mismatches1.remove(j)
                        # Salir del ciclo y continuar con otro nodo
                        break
    # Run over non matches neighbors
    for i in mismatches0:
        if i not in tracked:
            minadjdiff(i, mapping, tracked, depth + 1)

In [2]:
mol0 = read('test_ab_shuffled.xyz', index=0)
mol1 = read('test_ab_shuffled.xyz', index=1)
cutoffs0 = natural_cutoffs(mol0, mult=1.2)
cutoffs1 = natural_cutoffs(mol1, mult=1.2)
nl0 = NeighborList(cutoffs0, skin=0, self_interaction=False, bothways=True)
nl1 = NeighborList(cutoffs1, skin=0, self_interaction=False, bothways=True)
nl0.update(mol0)
nl1.update(mol1)
matrix0 = nl0.get_connectivity_matrix()
matrix1 = nl1.get_connectivity_matrix()
natom = len(mol0)
mapping = np.arange(natom)
print(f"There are {len(mol0)} atoms and {csgraph.connected_components(matrix0)[0]} molecule(s) in system 0")
print(f"There are {len(mol1)} atoms and {csgraph.connected_components(matrix1)[0]} molecule(s) in system 1")
print()
print(f"There are {moldiff(mapping)} initial differences with a distance of {rmsd(mapping):.4f}")
#print(triu(matrix0) != triu(matrix1))

minadjdiff(0, mapping, set())
print(f"There are {moldiff(mapping)} differences left after backtracking with a distance of {rmsd(mapping):.4f}")
write('output1.xyz', [mol0, mol1[mapping]])

There are 90 atoms and 1 molecule(s) in system 0
There are 90 atoms and 1 molecule(s) in system 1

There are 178 initial differences with a distance of 6.0366
There are 117 differences left after backtracking with a distance of 5.7986
